# BanglaHalluEval: SOMADHAN Reasoning Hallucination Pipeline
#
# Input CSV columns: question, answer, reasoning_len
#   - question      : the Bengali math problem
#   - answer        : full CoT chain + #### + final answer  (split at ####)
#   - reasoning_len : character length of the answer field
#
# Workflow:
# 1. Install dependencies
# 2. Mount Drive
# 3. Configure OpenAI API key
# 4. Upload SOMADHAN CSV
# 5. Select first 1200 rows (length-sorted input expected)
# 6. Assign 200 items per hallucination type (symmetric length coverage)
# 7. Generate hallucinated reasoning chains -> save CSV
# 8. Quality check
# 9. Download results

In [1]:
# @title 1. Install Dependencies
!pip -q install openai pandas

In [ ]:
# @title 2. Mount Drive
from google.colab import drive
from pathlib import Path
 
drive.mount('/content/drive')
 
BASE_DIR = '/content/drive/MyDrive/BanglaHalluEval'
Path(BASE_DIR).mkdir(parents=True, exist_ok=True)
print('Saving outputs to:', BASE_DIR)

In [ ]:
# @title 3. Configuration & API Setup
import csv
import json
import random
import time
from pathlib import Path
 
import pandas as pd
from google.colab import files, userdata
from openai import OpenAI
 
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    OPENAI_API_KEY = ''  # paste key here if not using Colab Secrets
 
if not OPENAI_API_KEY:
    print('ERROR: No API key found. Add it to Colab Secrets or paste directly above.')
else:
    print('API key configured.')
 
MODEL_NAME       = 'gpt-4o'  # change to gpt-5.4 when available on your account
RANDOM_SEED      = 42
CHECKPOINT_EVERY = 1          # save after every item (safe for long runs)
 
client = OpenAI(api_key=OPENAI_API_KEY)
 
DRIVE_OUT = Path(BASE_DIR) / 'Results' / 'reasoning_hallucination_results'
LOCAL_OUT  = Path('Results') / 'reasoning_hallucination_results'
for d in (DRIVE_OUT, LOCAL_OUT):
    d.mkdir(parents=True, exist_ok=True)
 
OUTPUT_1200         = DRIVE_OUT / 'somadhan_1200.csv'
OUTPUT_TYPED        = DRIVE_OUT / 'somadhan_1200_typed.csv'
OUTPUT_HALLUCINATED = DRIVE_OUT / 'somadhan_1200_hallucinated.csv'
LOG_PATH            = DRIVE_OUT / 'somadhan_1200_hallucinated.log'
 
OUTPUT_1200_LOCAL         = LOCAL_OUT / 'somadhan_1200.csv'
OUTPUT_TYPED_LOCAL        = LOCAL_OUT / 'somadhan_1200_typed.csv'
OUTPUT_HALLUCINATED_LOCAL = LOCAL_OUT / 'somadhan_1200_hallucinated.csv'
LOG_PATH_LOCAL            = LOCAL_OUT / 'somadhan_1200_hallucinated.log'

In [ ]:
# @title 4. Upload Dataset (SOMADHAN_SORTED.csv)
print("Please upload 'SOMADHAN_SORTED.csv'")
uploaded = files.upload()
INPUT_FILENAME = list(uploaded.keys())[0]
print(f'Uploaded: {INPUT_FILENAME}')

In [ ]:
# @title 5. Utility Functions
 
def load_rows(path):
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))
 
def write_rows(path, rows, fieldnames):
    with open(path, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
 
def write_rows_both(paths, rows, fieldnames):
    for p in paths:
        write_rows(str(p), rows, fieldnames)
 
def write_df_both(df, paths):
    for p in paths:
        df.to_csv(p, index=False, encoding='utf-8-sig')
 
def append_log(paths, text):
    for p in paths:
        with open(p, 'a', encoding='utf-8') as f:
            f.write(text)
 
def select_existing(primary, secondary):
    return primary if primary.exists() else secondary
 
def split_chain_answer(text):
    """
    Split the 'answer' column into (CoT chain, final answer).
    Column format:  <reasoning steps>  ####  <final answer>
    """
    cleaned = (text or '').strip()
    if '####' in cleaned:
        chain, answer = cleaned.rsplit('####', 1)
        return chain.strip(), answer.strip()
    return cleaned, ''
 
def assign_types_symmetric(df, type_order, per_type=200, seed=42):
    """
    Assign hallucination types so each type covers both short and long problems.
    Takes first (per_type//2) and last (per_type//2) from remaining pool per type.
    """
    df_out = df.copy().reset_index(drop=True)
    remaining = list(range(len(df_out)))
    assignments = {}
    half = per_type // 2
    rng = random.Random(seed)
 
    for t in type_order:
        if len(remaining) < per_type:
            rng.shuffle(remaining)
            selected = remaining[:per_type]
        else:
            selected = remaining[:half] + remaining[-half:]
        for idx in selected:
            assignments[idx] = t
        selected_set = set(selected)
        remaining = [i for i in remaining if i not in selected_set]
 
    df_out['hallucination_type'] = [assignments[i] for i in range(len(df_out))]
    return df_out
 
def parse_json_response(text):
    """Strip markdown fences and parse JSON. Returns (dict|None, error|None)."""
    cleaned = (text or '').strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.split('\n', 1)[-1].strip()
        if cleaned.endswith('```'):
            cleaned = cleaned.rsplit('```', 1)[0].strip()
    try:
        return json.loads(cleaned), None
    except Exception as e:
        return None, str(e)

In [ ]:
# @title 6. Select First 1200 Rows & Assign 6 Hallucination Types
 
df = pd.read_csv(INPUT_FILENAME)
df_1200 = df.head(1200).reset_index(drop=True)
write_df_both(df_1200, (OUTPUT_1200, OUTPUT_1200_LOCAL))
 
TYPE_ORDER = [
    'arithmetic_slip',
    'formula_misapplication',
    'variable_confusion',
    'invalid_deduction',
    'hallucinated_intermediate_fact',
    'semantic_drift',
]
 
df_typed = assign_types_symmetric(df_1200, TYPE_ORDER, per_type=200, seed=RANDOM_SEED)
write_df_both(df_typed, (OUTPUT_TYPED, OUTPUT_TYPED_LOCAL))
 
print('Saved:', OUTPUT_1200)
print('Saved:', OUTPUT_TYPED)
print('\nType distribution:')
print(df_typed['hallucination_type'].value_counts())

In [ ]:
# @title 7. Define Prompts (6 hallucination types)
 
# Rules are in English — LLMs follow English instructions more reliably.
# Bengali is used ONLY in the before/after examples so the model has a
# concrete reference in the target language.
# Section labels (Problem / Correct reasoning chain / Correct answer) stay English.
# Column names (question, answer, reasoning_len) are never translated.
 
BASE_PROMPT = """\
You are generating synthetic reasoning hallucinations for a Bengali NLP research dataset.
 
You will be given:
- A math word problem (in Bengali)
- A correct step-by-step reasoning chain (in Bengali, using <<expr=result>> annotations)
- The correct final answer
 
Your task:
Modify EXACTLY ONE reasoning step so the reasoning becomes incorrect but still appears plausible and coherent to a human reader.
 
Error type to apply:
<ERROR_TYPE_INSTRUCTION>
 
Requirements:
1. Modify only ONE reasoning step.
2. The error must be subtle and believable — not obviously absurd.
3. All steps after the modified step must remain internally consistent with the wrong value introduced.
4. Preserve the original Bengali wording style and sentence structure in all other steps.
5. The entire hallucinated_chain must be in Bengali.
6. Preserve the <<expr=result>> annotation format. In the modified step, both expr and result must reflect the wrong value (e.g. if you change 200 to 180, write <<২৫*৮=১৮০>>১৮০).
7. Do not switch any part of the output to English.
8. Do not add explanations or commentary outside the JSON.
9. hallucinated_answer should reflect the wrong final value that results from the error.
 
Return ONLY a JSON object in this exact format:
{{
  "hallucinated_chain": "...",
  "error_step": <integer step number>,
  "error_type": "...",
  "hallucinated_answer": "..."
}}
 
Problem:
<PROBLEM>
 
Correct reasoning chain:
<CHAIN>
 
Correct answer:
<ANSWER>
 
Return only the JSON object."""
 
# Per-type instruction blocks.
# Instructions are in English. Bengali appears only in the examples.
 
TYPE_INSTRUCTIONS = {
 
    'arithmetic_slip': """\
Type: Arithmetic Slip
Keep the logic structure completely intact. Insert one subtle numerical error in a single calculation.
The wrong number should be close to the correct one (roughly 5% to 20% off) — not obviously wrong.
All subsequent steps must use the wrong number consistently.
 
Example —
Correct step:     ২৫ দিন * ৮ ঘন্টা/দিন = <<২৫*৮=২০০>>২০০ ঘন্টা
Hallucinated:     ২৫ দিন * ৮ ঘন্টা/দিন = <<২৫*৮=১৮০>>১৮০ ঘন্টা
 
Note: Only the <<expr=result>> annotation and the inline number change.
The surrounding Bengali sentence stays identical. All later steps use ১৮০ instead of ২০০.""",
 
    'formula_misapplication': """\
Type: Formula Misapplication
Use a related but wrong operation (e.g. divide instead of multiply, add instead of multiply).
The Bengali narrative should justify the wrong operation so it still sounds plausible.
All subsequent steps must be consistent with the wrong result.
 
Example —
Correct step:     ২০০ ঘন্টা * ৳১৫/ঘন্টা = ৳<<২০০*১৫=৩০০০>>৩০০০
Hallucinated:     মোট আয় বের করতে ঘন্টার সংখ্যাকে ঘন্টার হার দিয়ে ভাগ করুন: ২০০ ÷ ১৫ = ৳<<২০০/১৫=১৩>>১৩
 
Note: The Bengali text now says "ভাগ করুন" (divide) to justify the wrong operation.
All later steps use ১৩ as the per-worker earning.""",
 
    'variable_confusion': """\
Type: Variable Confusion
Swap the values or meanings of two quantities from the problem (e.g. swap two rates, two counts, or two entities).
All subsequent steps must be consistent with the swapped values.
 
Example —
Problem context: ৪ জন গুদাম কর্মী ৳১৫/ঘন্টা, ২ জন ম্যানেজার ৳২০/ঘন্টা
Correct step:     ২০০ ঘন্টা * ৳১৫/ঘন্টা = ৳<<২০০*১৫=৩০০০>>৩০০০  (warehouse worker rate)
Hallucinated:     প্রতিটি গুদাম কর্মীর ঘন্টার হার ৳২০ হিসেবে গণনা করুন: ২০০ ঘন্টা * ৳২০/ঘন্টা = ৳<<২০০*২০=৪০০০>>৪,০০০
 
Note: Warehouse rate (৳১৫) and manager rate (৳২০) are swapped.
All subsequent steps use the swapped values.""",
 
    'invalid_deduction': """\
Type: Invalid Deduction
Inject one logically unsupported conclusion that does not actually follow from the previous step.
The transition should sound natural in Bengali but hide a logical leap not justified by the problem.
All subsequent steps must follow from this invalid conclusion.
 
Example —
Previous result: মোট মজুরি = ৳২০,০০০
Hallucinated step: যেহেতু কর্মচারীর সংখ্যা জোড় (৬ জন), তাই কর গণনার ক্ষেত্রে প্রতিটি জুটির জন্য আলাদাভাবে হিসাব করতে হবে। প্রতি জুটির মজুরি = ৳২০,০০০ ÷ ৩ = <<২০০০০/৩=৬৬৬৭>>৳৬,৬৬৭।
 
Note: Dividing by number of pairs is not stated anywhere — it is a hallucinated logical step.
All later steps use ৳৬,৬৬৭ as the base.""",
 
    'hallucinated_intermediate_fact': """\
Type: Hallucinated Intermediate Fact
Introduce one fabricated assumption not present anywhere in the problem.
The assumption must sound like a natural reading of the problem context.
All subsequent steps must proceed consistently from this fabricated fact.
 
Example —
Hallucinated step: প্রশ্নটি ইঙ্গিত করে যে কর্মীরা প্রতি মাসে ২ দিন ছুটি ভোগ করেন, তাই কার্যকর কাজের দিন = ২৫ - ২ = <<২৫-২=২৩>>২৩ দিন।
 
Note: "2 days off" appears nowhere in the problem — entirely fabricated but sounds plausible.
All later steps use ২৩ working days instead of ২৫.""",
 
    'semantic_drift': """\
Type: Semantic Drift
Slightly reinterpret what the problem is asking at some point mid-reasoning.
For example: the problem asks for total cost (wages + tax) but reasoning drifts to treating wages alone as the answer — or vice versa.
The drift should feel like a plausible reading of the problem, not an obvious mistake.
 
Example —
Problem asks for: total wages AND tax combined
Hallucinated step: যেহেতু প্রশ্নটি মূলত মজুরির মোট পরিমাণ জিজ্ঞেস করছে, তাই মোট মজুরি ৳২০,০০০ই চূড়ান্ত উত্তর। ট্যাক্সের পরিমাণ নিয়োগকর্তার নিজস্ব ব্যয় হিসেবে আলাদা ধরা হয়।
 
Note: The reasoning reinterprets "wages + tax" as "wages only", skipping the tax step entirely.
hallucinated_answer becomes ২০০০০ instead of ২২০০০.""",
}
 
PROMPT_MAP = {
    t: BASE_PROMPT.replace('<ERROR_TYPE_INSTRUCTION>', instr)
    for t, instr in TYPE_INSTRUCTIONS.items()
}
 
def build_prompt(h_type, problem, chain, answer):
    return (PROMPT_MAP[h_type]
            .replace('<PROBLEM>', problem)
            .replace('<CHAIN>', chain)
            .replace('<ANSWER>', answer))

In [ ]:
# @title 8. Run Generation Loop
 
typed_path = select_existing(OUTPUT_TYPED, OUTPUT_TYPED_LOCAL)
rows_df    = pd.read_csv(typed_path)
output_rows = []
completed_source_ids = set()
 
FIELDNAMES = [
    'id',
    'source_id',
    'hallucination_type',
    'question',
    'answer',
    'reasoning_len',
    'hallucinated_chain',
    'error_step',
    'error_type',
    'hallucinated_answer',
    'raw_response',
    'parse_error',
]
 
existing_path = select_existing(OUTPUT_HALLUCINATED, OUTPUT_HALLUCINATED_LOCAL)
if existing_path.exists():
    output_rows = load_rows(str(existing_path))
    completed_source_ids = {r.get('source_id') for r in output_rows if r.get('source_id')}
    print(f'Resuming: {len(completed_source_ids)} items already done.')
 
log_paths = (LOG_PATH, LOG_PATH_LOCAL)
append_log(log_paths, 'Starting reasoning hallucination generation.\n')
append_log(log_paths, f'Total items: {len(rows_df)}\n')
 
for idx, row in rows_df.iterrows():
    source_id = str(row['id']) if 'id' in rows_df.columns else str(idx + 1)
 
    if source_id in completed_source_ids:
        continue
 
    question      = str(row.get('question', '')).strip()
    full_answer   = str(row.get('answer', '')).strip()
    reasoning_len = row.get('reasoning_len', '')
    h_type        = row.get('hallucination_type', '')
 
    chain, correct_answer = split_chain_answer(full_answer)
    prompt = build_prompt(h_type, question, chain, correct_answer)
 
    raw_response = ''
    parsed       = None
    parse_error  = None
 
    try:
        response = client.responses.create(
            model=MODEL_NAME,
            input=[{'role': 'user', 'content': prompt}],
            max_output_tokens=512,
            temperature=0.7,
        )
        raw_response = response.output_text
        parsed, parse_error = parse_json_response(raw_response)
 
    except Exception as e:
        parse_error = f'API error: {e}'
        append_log(log_paths, f'[{idx+1}] API error for source_id={source_id}: {e}\n')
 
    output_rows.append({
        'id':                  f"{source_id}::{h_type}",
        'source_id':           source_id,
        'hallucination_type':  h_type,
        'question':            question,
        'answer':              full_answer,
        'reasoning_len':       reasoning_len,
        'hallucinated_chain':  (parsed or {}).get('hallucinated_chain', ''),
        'error_step':          (parsed or {}).get('error_step', ''),
        'error_type':          (parsed or {}).get('error_type', ''),
        'hallucinated_answer': (parsed or {}).get('hallucinated_answer', ''),
        'raw_response':        raw_response,
        'parse_error':         parse_error or '',
    })
 
    if (len(output_rows) % CHECKPOINT_EVERY) == 0:
        write_rows_both(
            (OUTPUT_HALLUCINATED, OUTPUT_HALLUCINATED_LOCAL),
            output_rows, FIELDNAMES
        )
        append_log(log_paths,
            f'[{idx+1}/{len(rows_df)}] source_id={source_id} | type={h_type} | saved {len(output_rows)} rows\n')
        print(f'[{idx+1}/{len(rows_df)}] {h_type} | saved {len(output_rows)} rows')
 
    time.sleep(0.1)
 
write_rows_both(
    (OUTPUT_HALLUCINATED, OUTPUT_HALLUCINATED_LOCAL),
    output_rows, FIELDNAMES
)
print(f'\nDone. Total rows written: {len(output_rows)}')
print(f'Output: {OUTPUT_HALLUCINATED}')
print(f'Log:    {LOG_PATH}')

In [ ]:
# @title 9. Quality Check
 
df_out = pd.read_csv(select_existing(OUTPUT_HALLUCINATED, OUTPUT_HALLUCINATED_LOCAL))
 
print('=== Total rows:', len(df_out))
print('\n--- Type distribution ---')
print(df_out['hallucination_type'].value_counts())
 
parse_errors = df_out[df_out['parse_error'].astype(str).str.len() > 0]
print(f'\n--- Parse errors: {len(parse_errors)} rows ---')
if len(parse_errors) > 0:
    print(parse_errors[['source_id', 'hallucination_type', 'parse_error']].head(10))
 
empty_chains = df_out[df_out['hallucinated_chain'].astype(str).str.len() == 0]
print(f'\n--- Empty hallucinated_chain: {len(empty_chains)} rows ---')
 
print('\n--- Sample output (first row) ---')
sample = df_out.iloc[0]
print('Type:             ', sample['hallucination_type'])
print('Error step:       ', sample['error_step'])
print('Error type:       ', sample['error_type'])
print('Hallucinated ans: ', sample['hallucinated_answer'])
print('Chain preview:    ', str(sample['hallucinated_chain'])[:300])

In [ ]:
# @title 10. Download Results
files.download(str(OUTPUT_1200))
files.download(str(OUTPUT_TYPED))
files.download(str(OUTPUT_HALLUCINATED))

In [1]:
# @title 1. Install Dependencies
!pip -q install openai pandas

In [ ]:
# @title 3. Configuration & API Setup
import csv
import json
import random
import time
from pathlib import Path

import pandas as pd
from google.colab import files, userdata
from openai import OpenAI

# @markdown Enter your OpenAI API Key below:
try:
    # Check if key is in Colab Secrets
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    OPENAI_API_KEY = ''  # @param {type:'string'}

if not OPENAI_API_KEY:
    print('ERROR: Please enter your OpenAI API Key in the field above.')
else:
    print('API Key configured.')

# @markdown Default Model for this task:
MODEL_NAME = 'gpt-5.4'  # @param {type:'string'}

client = OpenAI(api_key=OPENAI_API_KEY)

RANDOM_SEED = 42
CHECKPOINT_EVERY = 1

# Output files (saved in Drive + locally)
DRIVE_OUTPUT_DIR = Path(BASE_DIR) / 'Results' / 'reasoning_hallucination_results'
LOCAL_OUTPUT_DIR = Path('Results') / 'reasoning_hallucination_results'
for _dir in (DRIVE_OUTPUT_DIR, LOCAL_OUTPUT_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

OUTPUT_1200 = DRIVE_OUTPUT_DIR / 'somadhan_1200.csv'
OUTPUT_TYPED = DRIVE_OUTPUT_DIR / 'somadhan_1200_typed.csv'
OUTPUT_HALLUCINATED = DRIVE_OUTPUT_DIR / 'somadhan_1200_hallucinated.csv'
LOG_PATH = DRIVE_OUTPUT_DIR / 'somadhan_1200_hallucinated.log'

OUTPUT_1200_LOCAL = LOCAL_OUTPUT_DIR / 'somadhan_1200.csv'
OUTPUT_TYPED_LOCAL = LOCAL_OUTPUT_DIR / 'somadhan_1200_typed.csv'
OUTPUT_HALLUCINATED_LOCAL = LOCAL_OUTPUT_DIR / 'somadhan_1200_hallucinated.csv'
LOG_PATH_LOCAL = LOCAL_OUTPUT_DIR / 'somadhan_1200_hallucinated.log'

In [ ]:
# @title 5. Utility Functions

def load_rows(csv_path: str) -> list[dict]:
    with open(csv_path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

def split_chain_answer(text: str) -> tuple[str, str]:
    cleaned = (text or '').strip()
    if '####' in cleaned:
        chain, answer = cleaned.rsplit('####', 1)
        return chain.strip(), answer.strip()
    return cleaned, ''

def assign_types_symmetric(df: pd.DataFrame, type_order: list[str], per_type: int = 200, seed: int = 42) -> pd.DataFrame:
    df_out = df.copy().reset_index(drop=True)
    remaining = list(range(len(df_out)))
    assignments = {}
    half = per_type // 2

    for t in type_order:
        if len(remaining) < per_type:
            rng = random.Random(seed)
            rng.shuffle(remaining)
            selected = remaining[:per_type]
        else:
            selected = remaining[:half] + remaining[-half:]

        for idx in selected:
            assignments[idx] = t
        selected_set = set(selected)
        remaining = [i for i in remaining if i not in selected_set]

    df_out['hallucination_type'] = [assignments[i] for i in range(len(df_out))]
    return df_out

def build_prompt(template: str, problem: str, chain: str, answer: str) -> str:
    return template.replace('<PROBLEM>', problem).replace('<CHAIN>', chain).replace('<ANSWER>', answer)

def parse_json_response(text: str) -> tuple[dict | None, str | None]:
    cleaned = (text or '').strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.split('\n', 1)[-1].strip()
        if cleaned.endswith('```'):
            cleaned = cleaned.rsplit('```', 1)[0].strip()
    try:
        return json.loads(cleaned), None
    except Exception as e:
        return None, str(e)

def write_rows(filename: str, rows: list[dict], fieldnames: list[str]) -> None:
    with open(filename, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def write_rows_both(paths: tuple[Path, Path], rows: list[dict], fieldnames: list[str]) -> None:
    for path in paths:
        write_rows(str(path), rows, fieldnames)

def write_df_both(df: pd.DataFrame, paths: tuple[Path, Path]) -> None:
    df.to_csv(paths[0], index=False, encoding='utf-8-sig')
    df.to_csv(paths[1], index=False, encoding='utf-8-sig')

def select_existing_path(primary: Path, secondary: Path) -> Path:
    if primary.exists():
        return primary
    if secondary.exists():
        return secondary
    return primary

def append_log(paths: tuple[Path, Path], text: str) -> None:
    for path in paths:
        with open(path, 'a', encoding='utf-8') as log:
            log.write(text)

In [ ]:
# @title 7. Define Prompts

BASE_PROMPT = (
    "You are generating synthetic reasoning hallucinations for a research dataset. All the the texts should be in bangla\n"
    "Given:\n"
    "a math problem\n"
    "a correct reasoning chain\n"
    "a correct final answer\n"
    "Your task:\n"
    "Modify EXACTLY ONE reasoning step so that the reasoning becomes incorrect, but still appears plausible and coherent to a human reader.\n"
    "Requirements:\n"
    "1. Change only ONE reasoning step.\n"
    "2. The error should be subtle and believable.\n"
    "3. Do NOT make the reasoning obviously absurd.\n"
    "4. All later reasoning steps must remain internally consistent with the modified step.\n"
    "5. Maintain the original reasoning style and tone.\n"
    "6. The final answer should usually become incorrect.\n"
    "7. Do not add explanations or commentary.\n"
    "Possible error types:\n"
    "arithmetic slip\n"
    "formula misuse\n"
    "variable confusion\n"
    "invalid logical deduction\n"
    "hallucinated intermediate fact\n"
    "semantic drift\n"
    "Error type instructions:\n"
    "<ERROR_TYPE>\n"
    "Return output in JSON format:\n"
    "{\n"
    "\"hallucinated_chain\": \"...\",\n"
    "\"error_step\": <step number>,\n"
    "\"error_type\": \"...\",\n"
    "\"hallucinated_answer\": \"...\"\n"
    "}\n"
    "Problem:\n"
    "<PROBLEM>\n"
    "Correct reasoning chain:\n"
    "<CHAIN>\n"
    "Correct answer:\n"
    "<ANSWER>\n"
    "Return only the JSON object.\n"
    ""
)

ERROR_TYPE_INSTRUCTIONS = {
    'arithmetic_slip': 'Keep logic structure intact but insert a subtle numerical error.',
    'formula_misapplication': 'Use a related but wrong formula (e.g., multiply instead of divide).',
    'variable_confusion': 'Swap meanings of variables or quantities.',
    'invalid_deduction': 'Inject a logically unsupported transition.',
    'hallucinated_intermediate_fact': 'Introduce a fabricated assumption and proceed consistently.',
    'semantic_drift': 'Slightly reinterpret the task midway (e.g., speed vs acceleration).',
}

PROMPT_MAP = {
    t: BASE_PROMPT.replace('<ERROR_TYPE>', ERROR_TYPE_INSTRUCTIONS[t])
    for t in TYPE_ORDER
}

In [ ]:
# @title 9. Download Results
files.download(str(OUTPUT_1200))
files.download(str(OUTPUT_TYPED))
files.download(str(OUTPUT_HALLUCINATED))